Celda 1: Instalación e Importación de Librerías

In [1]:
# ==============================================================================
# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
!pip install yfinance tensorflow scikit-learn --quiet

import numpy as np
import pandas as pd
import yfinance as yf
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, GRU, SimpleRNN, Dropout

# Configurar semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


Celda 2: Ingesta de Datos y Creación de Indicadores Técnicos

In [2]:
# ==============================================================================
# 2. INGESTA DE DATOS Y FEATURE ENGINEERING
# ==============================================================================
ticker = "FSM"  # Puedes cambiar a VOLCABC1.LM, ABX.TO, BVN, BHP
print("descargando datos para {ticker}...")

# Descargar últimos 2 años de datos reales
datos = yf.download(ticker, period="2y", interval="1d")

# Calcular indicadores técnicos base
datos['SMA_10'] = datos['Close'].rolling(window=10).mean()
datos['EMA_20'] = datos['Close'].rolling(window=20).mean()

# Definir la variable objetivo (Clasificación Binaria: 1 si sube mañana, 0 si baja/igual)
datos['Target'] = (datos['Close'].shift(-1) > datos['Close']).astype(int)

# Limpiar valores nulos generados por ventanas e indicadores
datos.dropna(inplace=True)

# Seleccionar características para el modelo de IA
features = ['Close', 'SMA_10', 'EMA_20']
X_raw = datos[features].values
y_raw = datos['Target'].values

print(f"Datos listos de {ticker}. Total de registros disponibles: {len(datos)}")

descargando datos para {ticker}...


/tmp/ipykernel_1596/3390835775.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos = yf.download(ticker, period="2y", interval="1d")
[*********************100%***********************]  1 of 1 completed

Datos listos de FSM. Total de registros disponibles: 482


Celda 3: Creación de Ventanas Temporales y Escalado

In [3]:
# ==============================================================================
# 3. CREACIÓN DE VENTANAS DESLIZANTES Y DIVISIÓN DE DATOS
# ==============================================================================
VENTANA = 10  # Usar los últimos 10 días para predecir el día siguiente

X_ventanas = []
y_ventanas = []

for i in range(len(X_raw) - VENTANA):
    X_ventanas.append(X_raw[i : i + VENTANA])
    y_ventanas.append(y_raw[i + VENTANA])

X_ventanas = np.array(X_ventanas)
y_ventanas = np.array(y_ventanas)

# Escalado de datos tridimensionales (Muestras, Pasos de tiempo, Características)
scaler = MinMaxScaler(feature_range=(0, 1))
# Ajustamos el escalador sobre las 2 dimensiones planas colapsando temporalmente
X_ventanas_scaled = X_ventanas.reshape(-1, X_ventanas.shape[-1])
X_ventanas_scaled = scaler.fit_transform(X_ventanas_scaled)
X_ventanas_scaled = X_ventanas_scaled.reshape(X_ventanas.shape)

# División de datos: 80% Entrenamiento, 20% Prueba (Sin mezclar por ser serie de tiempo)
split = int(len(X_ventanas_scaled) * 0.8)
X_train, X_test = X_ventanas_scaled[:split], X_ventanas_scaled[split:]
y_train, y_test = y_ventanas[:split], y_ventanas[split:]

print(f"Forma de X_train: {X_train.shape} | Forma de X_test: {X_test.shape}")

Forma de X_train: (377, 10, 3) | Forma de X_test: (95, 10, 3)


Celda 4: Entrenamiento y Evaluación de las 4 Arquitecturas

In [4]:
# ==============================================================================
# 4. CONSTRUCCIÓN, ENTRENAMIENTO Y EVALUACIÓN MULTI-ARQUITECTURA
# ==============================================================================
def evaluar_modelo(y_real, y_pred_prob):
    y_pred = (y_pred_prob > 0.5).astype(int)
    cm = confusion_matrix(y_real, y_pred)
    return {
        "accuracy": round(accuracy_score(y_real, y_pred), 4),
        "precision": round(precision_score(y_real, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_real, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_real, y_pred, zero_division=0), 4),
        "matrix": [int(cm[0][0]), int(cm[0][1]), int(cm[1][0]), int(cm[1][1])] # TN, FP, FN, TP
    }

resultados = {}
ultimas_predicciones = {}

# --- 4.1 ARQUITECTURA LSTM ---
print("\n--- Entrenando LSTM ---")
model_lstm = Sequential([
    LSTM(260, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(130),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train, y_train, epochs=80, batch_size=64, verbose=0)
pred_lstm = model_lstm.predict(X_test).flatten()
resultados["lstm"] = evaluar_modelo(y_test, pred_lstm)
ultimas_predicciones["lstm"] = float(pred_lstm[-1])

# --- 4.2 ARQUITECTURA BiLSTM ---
print("--- Entrenando BiLSTM ---")
model_bilstm = Sequential([
    Bidirectional(LSTM(100, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Bidirectional(LSTM(50)),
    Dense(1, activation='sigmoid')
])
model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bilstm.fit(X_train, y_train, epochs=80, batch_size=64, verbose=0)
pred_bilstm = model_bilstm.predict(X_test).flatten()
resultados["bilstm"] = evaluar_modelo(y_test, pred_bilstm)
ultimas_predicciones["bilstm"] = float(pred_bilstm[-1])

# --- 4.3 ARQUITECTURA GRU ---
print("--- Entrenando GRU ---")
model_gru = Sequential([
    GRU(280, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    GRU(140),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.fit(X_train, y_train, epochs=80, batch_size=64, verbose=0)
pred_gru = model_gru.predict(X_test).flatten()
resultados["gru"] = evaluar_modelo(y_test, pred_gru)
ultimas_predicciones["gru"] = float(pred_gru[-1])

# --- 4.4 ARQUITECTURA SimpleRNN ---
print("--- Entrenando SimpleRNN ---")
model_rnn = Sequential([
    SimpleRNN(180, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    SimpleRNN(90),
    Dense(1, activation='sigmoid')
])
model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.fit(X_train, y_train, epochs=80, batch_size=64, verbose=0)
pred_rnn = model_rnn.predict(X_test).flatten()
resultados["simplernn"] = evaluar_modelo(y_test, pred_rnn)
ultimas_predicciones["simplernn"] = float(pred_rnn[-1])

print("\n¡Entrenamiento completo de todas las RNNs!")


--- Entrenando LSTM ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 157ms/step
--- Entrenando BiLSTM ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 303ms/step
--- Entrenando GRU ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 191ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- Entrenando SimpleRNN ---
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step

¡Entrenamiento completo de todas las RNNs!


Celda 5: Generación y Exportación del Contrato de Datos JSON

In [ ]:
# ==============================================================================
# 5. CONTRATO DE DATOS JSON PARA EL FRONTEND
# ==============================================================================
# Helper para definir la señal del semáforo basada en la probabilidad final
def obtener_senal(probabilidad):
    if probabilidad > 0.65: return "BUY"
    elif probabilidad < 0.35: return "SELL"
    return "HOLD"

contrato_rnns = {
    "ticker": ticker,
    "ultima_actualizacion": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    "modelos": {
        "lstm": {
            "metricas": resultados["lstm"],
            "probabilidad_mañana": ultimas_predicciones["lstm"],
            "senal": obtener_senal(ultimas_predicciones["lstm"])
        },
        "bilstm": {
            "metricas": resultados["bilstm"],
            "probabilidad_mañana": ultimas_predicciones["bilstm"],
            "senal": obtener_senal(ultimas_predicciones["bilstm"])
        },
        "gru": {
            "metricas": resultados["gru"],
            "probabilidad_mañana": ultimas_predicciones["gru"],
            "senal": obtener_senal(ultimas_predicciones["gru"])
        },
        "simplernn": {
            "metricas": resultados["simplernn"],
            "probabilidad_mañana": ultimas_predicciones["simplernn"],
            "senal": obtener_senal(ultimas_predicciones["simplernn"])
        }
    }
}

# Guardar archivo local
with open('datos_rnns_clf.json', 'w') as f:
    json.dump(contrato_rnns, f, indent=4)

print("Archivo 'datos_rnns_clf.json' generado con éxito con la estructura requerida.")

Archivo 'datos_rnns_clf.json' generado con éxito con la estructura requerida.
